# A/B Test: упрощение checkout в e-commerce

## 1. Бизнес-контекст и гипотеза

Команда e-commerce продукта рассматривает упрощение процесса оформления заказа

Текущий checkout содержит несколько последовательных шагов. Предполагается, что сокращение количества действий может уменьшить потери пользователей перед покупкой

В рамках A/B-теста:

- **A — control:** текущая версия checkout
- **B — treatment:** упрощённая версия checkout

### Продуктовая гипотеза

Если упростить процесс оформления заказа, то конверсия пользователя в покупку увеличится, поскольку пользователю потребуется меньше действий для завершения заказа

### Основная метрика

**Conversion Rate в покупку**

### Дополнительная метрика

**ARPU** — средняя выручка на пользователя

### Guardrail-метрика

**Checkout Error Rate** — доля пользователей, столкнувшихся с технической ошибкой во время оформления

Новый вариант можно рекомендовать к запуску, если основная метрика статистически значимо улучшится и при этом guardrail-метрика не покажет значимого ухудшения

## 2. Статистическая постановка

Для основной метрики проверим:

**H0:** конверсия групп A и B не различается

**H1:** конверсия групп A и B различается

Уровень значимости:

`α = 0.05`

Если `p-value < 0.05`, нулевая гипотеза будет отвергнута

Дополнительно рассчитаем 95% доверительный интервал для разницы конверсий

Важно: статистическая значимость сама по себе не означает продуктовую полезность. Поэтому кроме `p-value` необходимо оценивать величину эффекта и его бизнес-смысл

## 3. Данные эксперимента

Для демонстрации полного процесса A/B-анализа используется синтетический экспериментальный датасет

Данные генерируются с фиксированным `random seed`, поэтому результаты анализа воспроизводимы

In [1]:
%pip install scipy

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm, ttest_ind

np.random.seed(42)

In [3]:
rng = np.random.default_rng(42)

n_per_group = 12_000

groups = np.array(
    ['A'] * n_per_group +
    ['B'] * n_per_group
)

# Вероятность покупки
converted = np.concatenate([
    rng.binomial(1, 0.120, n_per_group),
    rng.binomial(1, 0.133, n_per_group)
])

# Выручка только для совершивших покупку
mu = np.log(600) - 0.55**2 / 2

revenue = np.zeros(2 * n_per_group)

buyers_idx = np.where(converted == 1)[0]

revenue[buyers_idx] = rng.lognormal(
    mean=mu,
    sigma=0.55,
    size=len(buyers_idx)
)

# Технические ошибки checkout
checkout_error = np.concatenate([
    rng.binomial(1, 0.020, n_per_group),
    rng.binomial(1, 0.021, n_per_group)
])

ab = pd.DataFrame({
    'group': groups,
    'converted': converted,
    'revenue': revenue,
    'checkout_error': checkout_error
})

ab.head()

,group,converted,revenue,checkout_error
0,A,0,0.0,0
1,A,0,0.0,0
2,A,0,0.0,0
3,A,0,0.0,0
4,A,0,0.0,0


In [4]:
ab.groupby('group').agg(
    users=('converted', 'size'),
    conversions=('converted', 'sum'),
    conversion_rate=('converted', 'mean'),
    arpu=('revenue', 'mean'),
    checkout_error_rate=('checkout_error', 'mean')
).round(4)

,users,conversions,conversion_rate,arpu,checkout_error_rate
group,,,,,
A,12000,1393,0.1161,72.0198,0.0194
B,12000,1607,0.1339,81.8236,0.0202


## 4. Анализ основной метрики — Conversion Rate

Для сравнения двух долей используем z-тест

Кроме статистической значимости оценим:

- абсолютный uplift в процентных пунктах
- относительный uplift
- 95% доверительный интервал разницы

In [5]:
group_a = ab[ab['group'] == 'A']
group_b = ab[ab['group'] == 'B']

n_a = len(group_a)
n_b = len(group_b)

x_a = group_a['converted'].sum()
x_b = group_b['converted'].sum()

cr_a = x_a / n_a
cr_b = x_b / n_b

absolute_uplift = cr_b - cr_a
relative_uplift = absolute_uplift / cr_a

print('CR A:', round(cr_a * 100, 2), '%')
print('CR B:', round(cr_b * 100, 2), '%')
print('Absolute uplift:', round(absolute_uplift * 100, 2), 'p.p.')
print('Relative uplift:', round(relative_uplift * 100, 2), '%')

CR A: 11.61 %
CR B: 13.39 %
Absolute uplift: 1.78 p.p.
Relative uplift: 15.36 %


In [6]:
p_pool = (x_a + x_b) / (n_a + n_b)

se_pool = np.sqrt(
    p_pool * (1 - p_pool) *
    (1 / n_a + 1 / n_b)
)

z_stat = (cr_b - cr_a) / se_pool

p_value = 2 * (
    1 - norm.cdf(abs(z_stat))
)

print('z-statistic:', round(z_stat, 3))
print('p-value:', round(p_value, 6))

z-statistic: 4.177
p-value: 3e-05


In [7]:
se_diff = np.sqrt(
    cr_a * (1 - cr_a) / n_a +
    cr_b * (1 - cr_b) / n_b
)

ci_low = absolute_uplift - 1.96 * se_diff
ci_high = absolute_uplift + 1.96 * se_diff

print(
    '95% CI:',
    round(ci_low * 100, 2),
    '—',
    round(ci_high * 100, 2),
    'p.p.'
)

95% CI: 0.95 — 2.62 p.p.


**Вывод**

Конверсия группы B выше конверсии группы A примерно на **1.78 процентного пункта**, что соответствует относительному росту около **15.4%**

`p-value < 0.05`, поэтому нулевая гипотеза об отсутствии различий отвергается

95% доверительный интервал для эффекта составляет примерно **+0.95 — +2.62 п.п.** и не включает 0

Таким образом, рост Conversion Rate является статистически значимым

При этом для продуктового решения важно учитывать не только наличие статистической значимости, но и размер эффекта, а также дополнительные и guardrail-метрики

## 5. Дополнительная метрика — ARPU

Проверим, сопровождается ли рост конверсии ростом средней выручки на пользователя

ARPU рассчитывается по всем пользователям группы, включая тех, кто ничего не купил

In [8]:
arpu_a = group_a['revenue'].mean()
arpu_b = group_b['revenue'].mean()

arpu_diff = arpu_b - arpu_a
arpu_uplift = arpu_diff / arpu_a

print('ARPU A:', round(arpu_a, 2))
print('ARPU B:', round(arpu_b, 2))
print('Absolute difference:', round(arpu_diff, 2))
print('Relative uplift:', round(arpu_uplift * 100, 2), '%')


ARPU A: 72.02
ARPU B: 81.82
Absolute difference: 9.8
Relative uplift: 13.61 %


In [9]:
t_stat, arpu_p_value = ttest_ind(
    group_b['revenue'],
    group_a['revenue'],
    equal_var=False
)

print('t-statistic:', round(t_stat, 3))
print('p-value:', round(arpu_p_value, 6))

t-statistic: 3.148
p-value: 0.001648


In [10]:
var_a = group_a['revenue'].var(ddof=1)
var_b = group_b['revenue'].var(ddof=1)

se_arpu = np.sqrt(
    var_a / n_a +
    var_b / n_b
)

df = (
    (var_a / n_a + var_b / n_b) ** 2
    /
    (
        (var_a / n_a) ** 2 / (n_a - 1)
        +
        (var_b / n_b) ** 2 / (n_b - 1)
    )
)

from scipy.stats import t

t_crit = t.ppf(0.975, df)

arpu_ci_low = arpu_diff - t_crit * se_arpu
arpu_ci_high = arpu_diff + t_crit * se_arpu

print(
    '95% CI:',
    round(arpu_ci_low, 2),
    '—',
    round(arpu_ci_high, 2)
)

95% CI: 3.7 — 15.91


**Вывод**

ARPU группы B выше ARPU группы A примерно на **9.80**, что соответствует относительному росту около **13.6%**

Различие статистически значимо: `p-value < 0.05`

95% доверительный интервал для разницы ARPU составляет примерно **+3.70 — +15.91** и не включает 0

Таким образом, рост основной метрики сопровождается ростом средней выручки на пользователя

## 6. Guardrail-метрика — Checkout Error Rate

Рост конверсии не должен достигаться ценой ухудшения технического качества checkout

Поэтому сравним долю пользователей, столкнувшихся с технической ошибкой

In [11]:
error_a = group_a['checkout_error'].sum()
error_b = group_b['checkout_error'].sum()

error_rate_a = error_a / n_a
error_rate_b = error_b / n_b

print('Error rate A:', round(error_rate_a * 100, 2), '%')
print('Error rate B:', round(error_rate_b * 100, 2), '%')
print(
    'Difference:',
    round((error_rate_b - error_rate_a) * 100, 2),
    'p.p.'
)

Error rate A: 1.94 %
Error rate B: 2.02 %
Difference: 0.08 p.p.


In [12]:
error_pool = (
    error_a + error_b
) / (
    n_a + n_b
)

error_se = np.sqrt(
    error_pool * (1 - error_pool)
    * (1 / n_a + 1 / n_b)
)

error_z = (
    error_rate_b - error_rate_a
) / error_se

error_p_value = 2 * (
    1 - norm.cdf(abs(error_z))
)

print('z-statistic:', round(error_z, 3))
print('p-value:', round(error_p_value, 6))

z-statistic: 0.417
p-value: 0.676608


**Вывод**

Checkout Error Rate составляет около **1.94%** в группе A и **2.02%** в группе B

Разница составляет примерно **+0.08 п.п.**

`p-value > 0.05`, поэтому статистически значимого ухудшения guardrail-метрики не обнаружено

Следовательно, по имеющимся данным рост конверсии не сопровождается подтверждённым ухудшением технического качества checkout

## 7. Статистическая мощность и MDE

Для A/B-теста важно заранее понимать, эффект какого размера эксперимент способен обнаружить

Используем:

- `α = 0.05`
- power = `80%`
- одинаковый размер групп
- baseline Conversion Rate группы A

In [13]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

In [14]:
%pip install statsmodels

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [15]:
power_analysis = NormalIndPower()

baseline_cr = cr_a

min_effect_size = power_analysis.solve_power(
    effect_size=None,
    nobs1=n_a,
    alpha=0.05,
    power=0.80,
    ratio=1,
    alternative='two-sided'
)

low = baseline_cr
high = min(0.999, baseline_cr + 0.20)

for _ in range(100):
    mid = (low + high) / 2

    effect = abs(
        proportion_effectsize(
            mid,
            baseline_cr
        )
    )

    if effect < min_effect_size:
        low = mid
    else:
        high = mid

detectable_cr = (low + high) / 2
mde = detectable_cr - baseline_cr

print('Baseline CR:', round(baseline_cr * 100, 2), '%')
print('MDE:', round(mde * 100, 2), 'p.p.')
print(
    'Relative MDE:',
    round(mde / baseline_cr * 100, 2),
    '%'
)

Baseline CR: 11.61 %
MDE: 1.18 p.p.
Relative MDE: 10.19 %


**Вывод**

При размере около 12 000 пользователей на группу эксперимент способен обнаружить изменение Conversion Rate примерно от **1.18 процентного пункта** при мощности 80%

Наблюдаемый эффект составляет около **1.78 п.п.**, то есть превышает рассчитанный MDE

Это означает, что эксперимент имеет достаточную чувствительность для обнаружения эффекта такого масштаба

В реальном эксперименте baseline Conversion Rate и MDE должны быть определены до запуска теста на основе исторических данных и бизнес-требований

В данном учебном кейсе текущий baseline используется для демонстрации расчёта чувствительности эксперимента

## 8. Ошибки I и II рода

При интерпретации результатов A/B-теста необходимо учитывать два типа статистических ошибок

**Ошибка I рода** — отклонить нулевую гипотезу, когда реального эффекта нет

Вероятность ошибки I рода контролируется уровнем значимости `α`

В данном эксперименте используется `α = 0.05`

**Ошибка II рода** — не обнаружить эффект, который на самом деле существует

Вероятность ошибки II рода обозначается `β`, а статистическая мощность определяется как:

`power = 1 - β`

При мощности 80% вероятность не обнаружить эффект заданного размера составляет около 20%

Поэтому размер выборки, MDE, уровень значимости и требуемая мощность должны определяться до запуска эксперимента

## 9. Итоговое продуктовое решение

Вариант B показывает улучшение основной продуктовой метрики

- Conversion Rate вырос с **11.61% до 13.39%**
- абсолютный uplift составил около **+1.78 п.п.**
- относительный uplift — около **+15.4%**
- эффект статистически значим
- 95% доверительный интервал не включает 0
- наблюдаемый эффект превышает рассчитанный MDE

Дополнительная метрика также улучшилась:

- ARPU вырос примерно с **72.02 до 81.82**
- относительный рост составляет около **13.6%**
- различие статистически значимо

Guardrail-метрика не показывает статистически подтверждённого ухудшения:

- Checkout Error Rate: **1.94% → 2.02%**
- `p-value = 0.677`

### Решение

По результатам эксперимента вариант B можно рекомендовать к постепенному rollout

Перед полным запуском стоит дополнительно проверить:

- стабильность эффекта по устройствам и каналам привлечения
- влияние на отмены и возвраты
- сохранение эффекта во времени
- техническую стабильность при увеличении нагрузки

Таким образом, решение принимается не только на основании `p-value`, но и с учётом размера эффекта, доверительного интервала, MDE, дополнительных метрик, guardrail-метрик и бизнес-контекста